In [21]:
import pandas as pd
import numpy as np

# ----------------------------------------------------------------------
# 0. CONFIG
# ----------------------------------------------------------------------
INPUT_FILE = r"C:\SuperMarket Analysis\data\SuperMarket_Analytics_raw_data.csv"
OUTPUT_FILE = r"C:\Users\Gowsalya\SuperMarket_Cleaned.csv"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

# 1. LOAD
df = pd.read_csv(INPUT_FILE)
print(f"Raw shape: {df.shape}")

# Keep an original copy for a before/after summary at the end
raw_rows = len(df)


# 2. STANDARDIZE COLUMN NAMES (safety net in case of stray spaces/case)
df.columns = (
    df.columns.str.strip()
    .str.replace(" ", "_")
)


# 3. STRIP WHITESPACE FROM ALL TEXT/OBJECT COLUMNS
obj_cols = df.select_dtypes(include=["object", "string"]).columns
for col in obj_cols:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({"nan": np.nan, "": np.nan, "None": np.nan})

# 4. REMOVE EXACT DUPLICATE ROWS AND DUPLICATE ORDER_IDs
before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} exact duplicate rows")

before = len(df)
df = df.drop_duplicates(subset="Order_ID", keep="first")
print(f"Removed {before - len(df)} duplicate Order_ID rows")


# 5. FIX DATA TYPES

# Order_Date: dd-mm-yyyy -> datetime
df["Order_Date"] = pd.to_datetime(df["Order_Date"], format="%Y-%m-%d", errors="coerce")

# Numeric columns
numeric_cols = ["Age", "Quantity", "Unit_Price", "Discount_Percent",
                 "Sales", "Cost", "Profit", "Rating"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Categorical/text normalization
df["Gender"] = df["Gender"].str.title().str.strip()
df["Gender"] = df["Gender"].replace({"M": "Male", "F": "Female"})

df["City"] = df["City"].str.title().str.strip()
df["Region"] = df["Region"].str.title().str.strip()
df["Product_Category"] = df["Product_Category"].str.title().str.strip()
df["Product_Name"] = df["Product_Name"].str.title().str.strip()
df["Payment_Mode"] = df["Payment_Mode"].str.upper().str.strip()
df["Payment_Mode"] = df["Payment_Mode"].replace({"UPI": "UPI", "CARD": "Card", "CASH": "Cash"})
df["Customer_Name"] = df["Customer_Name"].str.title().str.strip()


# 6. HANDLE MISSING VALUES
print("\nMissing values per column (before imputation):")
print(df.isna().sum()[df.isna().sum() > 0])

# Drop rows with no Order_ID or no Order_Date (unusable records)
df = df.dropna(subset=["Order_ID", "Order_Date"])

# Numeric fields: fill missing with median (robust to outliers)
for col in ["Age", "Unit_Price", "Discount_Percent", "Quantity"]:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

# Rating: fill missing with median rating, keep as integer 1-5
if df["Rating"].isna().any():
    df["Rating"] = df["Rating"].fillna(df["Rating"].median())
df["Rating"] = df["Rating"].round().clip(1, 5).astype(int)

# Categorical fields: fill with "Unknown"
for col in ["City", "Region", "Product_Category", "Product_Name",
            "Payment_Mode", "Gender", "Customer_Name"]:
    df[col] = df[col].fillna("Unknown")


# 7. VALIDATE / RECOMPUTE FINANCIAL FIELDS
df["Sales_Calculated"] = (
    df["Quantity"] * df["Unit_Price"] * (1 - df["Discount_Percent"] / 100)
).round(2)

# Flag rows where recorded Sales deviates materially (>1 rupee) from calculated Sales
df["Sales_Mismatch"] = (df["Sales"] - df["Sales_Calculated"]).abs() > 1

# If Sales is missing, fill it from the calculated value
df["Sales"] = df["Sales"].fillna(df["Sales_Calculated"])

# Recompute Profit = Sales - Cost where missing
df["Profit"] = df["Profit"].fillna(df["Sales"] - df["Cost"])

# Flag inconsistent Profit values (Sales - Cost != Profit, beyond rounding)
df["Profit_Mismatch"] = (df["Sales"] - df["Cost"] - df["Profit"]).abs() > 1

print(f"\nRows with Sales mismatch (recorded vs calculated): {df['Sales_Mismatch'].sum()}")
print(f"Rows with Profit mismatch (Sales-Cost vs Profit):    {df['Profit_Mismatch'].sum()}")


# 8. VALIDATE RANGES / REMOVE IMPOSSIBLE VALUES

# Age: keep plausible customer range
df = df[(df["Age"] >= 10) & (df["Age"] <= 100)]

# Quantity, Unit_Price, Sales, Cost must be non-negative
for col in ["Quantity", "Unit_Price", "Sales", "Cost"]:
    df = df[df[col] >= 0]

# Discount_Percent must be within 0-100
df = df[(df["Discount_Percent"] >= 0) & (df["Discount_Percent"] <= 100)]

# Rating must be within 1-5
df = df[(df["Rating"] >= 1) & (df["Rating"] <= 5)]


# 9. OUTLIER DETECTION (IQR method) — flagged, not removed
def flag_outliers_iqr(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - k * iqr, q3 + k * iqr
    return (series < lower) | (series > upper)

df["Sales_Outlier"] = flag_outliers_iqr(df["Sales"])
df["Profit_Outlier"] = flag_outliers_iqr(df["Profit"])

print(f"\nSales outliers flagged:  {df['Sales_Outlier'].sum()}")
print(f"Profit outliers flagged: {df['Profit_Outlier'].sum()}")


# 10. FEATURE ENGINEERING (handy derived columns for analysis)
df["Order_Year"] = df["Order_Date"].dt.year
df["Order_Month"] = df["Order_Date"].dt.month
df["Order_Month_Name"] = df["Order_Date"].dt.strftime("%b")
df["Order_Weekday"] = df["Order_Date"].dt.day_name()
df["Profit_Margin_%"] = ((df["Profit"] / df["Sales"].replace(0, np.nan)) * 100).round(2)


# 11. FINAL COLUMN ORDER & RESET INDEX
df = df.sort_values("Order_Date").reset_index(drop=True)


# 12. SUMMARY REPORT
print("\n" + "=" * 55)
print("CLEANING SUMMARY")
print("=" * 55)
print(f"Raw rows:     {raw_rows}")
print(f"Cleaned rows: {len(df)}")
print(f"Rows removed: {raw_rows - len(df)}")
print(f"Date range:   {df['Order_Date'].min().date()} to {df['Order_Date'].max().date()}")
print(f"Columns:      {list(df.columns)}")
print("\nData types:")
print(df.dtypes)

# 13. EXPORT
df.to_csv(OUTPUT_FILE, index=False)
print(f"\nCleaned data saved to: {OUTPUT_FILE}")

Raw shape: (1000, 18)
Removed 0 exact duplicate rows
Removed 0 duplicate Order_ID rows

Missing values per column (before imputation):
Series([], dtype: int64)

Rows with Sales mismatch (recorded vs calculated): 0
Rows with Profit mismatch (Sales-Cost vs Profit):    0

Sales outliers flagged:  15
Profit outliers flagged: 28

CLEANING SUMMARY
Raw rows:     1000
Cleaned rows: 1000
Rows removed: 0
Date range:   2024-01-05 to 2025-12-30
Columns:      ['Order_ID', 'Order_Date', 'Customer_ID', 'Customer_Name', 'Gender', 'Age', 'City', 'Region', 'Product_Category', 'Product_Name', 'Quantity', 'Unit_Price', 'Discount_Percent', 'Sales', 'Cost', 'Profit', 'Payment_Mode', 'Rating', 'Sales_Calculated', 'Sales_Mismatch', 'Profit_Mismatch', 'Sales_Outlier', 'Profit_Outlier', 'Order_Year', 'Order_Month', 'Order_Month_Name', 'Order_Weekday', 'Profit_Margin_%']

Data types:
Order_ID                       str
Order_Date          datetime64[us]
Customer_ID                    str
Customer_Name            

In [22]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\SuperMarket Analysis\data\SuperMarket_Analytics_raw_data.csv")
print("Raw rows:", len(df))

# Standardize columns
df.columns = df.columns.str.strip().str.replace(" ", "_")

# Check Order_Date parsing
df["Order_Date"] = pd.to_datetime(df["Order_Date"], format="%d-%m-%Y", errors="coerce")
print("Rows with valid Order_Date:", df["Order_Date"].notna().sum())
print("Rows with NaT (failed date parse):", df["Order_Date"].isna().sum())
print(df["Order_Date"].head())

Raw rows: 1000
Rows with valid Order_Date: 0
Rows with NaT (failed date parse): 1000
0   NaT
1   NaT
2   NaT
3   NaT
4   NaT
Name: Order_Date, dtype: datetime64[s]


In [23]:
df_raw = pd.read_csv(r"C:\SuperMarket Analysis\data\SuperMarket_Analytics_raw_data.csv")
print(df_raw["Order_Date"].head(10))
print(df_raw["Order_Date"].dtype)

0    2025-11-23
1    2024-07-22
2    2024-12-10
3    2025-01-22
4    2024-03-22
5    2024-09-30
6    2024-11-28
7    2025-05-26
8    2025-03-14
9    2025-03-08
Name: Order_Date, dtype: str
str


In [24]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce", dayfirst=True)
print("Valid dates:", df["Order_Date"].notna().sum())

Valid dates: 0
